###What is VACUUM in Delta Lake?

In Delta Lake, every operation (INSERT, UPDATE, DELETE, MERGE, OPTIMIZE)
creates new Parquet files and marks old ones as deleted in the _delta_log.

However, those old data files remain physically present in storage
(to support Time Travel and versioning).

✅ The VACUUM command physically removes these old, unreferenced files
from the storage to save space.

🔹 Why Use VACUUM?
Problem	Solution
Storage keeps increasing over time	Removes obsolete data files
Too many small files	Cleans up after OPTIMIZE
You don’t need old versions for time travel	Frees up storage space
🔹 Syntax
VACUUM [table_name | delta.`path`] [RETAIN num HOURS];


Parameters:

table_name → Registered Delta table

delta.path → File path to the Delta folder

RETAIN → Number of hours to keep old files (default: 168 hours = 7 days)

🔹 Example 1 — Simple VACUUM
VACUUM delta.`/delta/movies_zorder`;


This will delete all files older than 7 days.

🔹 Example 2 — With a Table Reference
VACUUM movies_zorder RETAIN 168 HOURS;


Deletes files not needed for table versioning older than 7 days.

🔹 Example 3 — Reduce Retention Period (⚠️ Be Careful)
VACUUM movies_zorder RETAIN 0 HOURS;


This immediately deletes all unreferenced files.

⚠️ Risk:
You cannot time travel or rollback after those files are deleted.
Delta Lake protects you by default —
it will throw an error unless you disable retention check (for testing only):

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", False)
spark.sql("VACUUM movies_zorder RETAIN 0 HOURS")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_practice.inputdb.employees_vac (
  id INT,
  name STRING,
  department STRING,
  salary DECIMAL(10, 2),
  hire_date DATE
) USING DELTA;


In [0]:
%sql
INSERT INTO databricks_practice.inputdb.employees_vac VALUES
  (1, 'Alice Smith', 'Engineering', 75000.00, '2022-01-15'),
  (2, 'Bob Johnson', 'Marketing', 60000.00, '2021-03-20'),
  (3, 'Charlie Brown', 'Sales', 80000.00, '2023-06-01'),
  (4, 'Diana Prince', 'Engineering', 90000.00, '2020-11-10');



In [0]:
%sql
UPDATE databricks_practice.inputdb.employees_vac
SET name = "Rajini"
WHERE id = 1;

delete from databricks_practice.inputdb.employees_vac where id =2;

In [0]:
%sql
DESCRIBE DETAIL databricks_practice.inputdb.employees_vac

In [0]:
%sql
INSERT INTO databricks_practice.inputdb.employees_vac VALUES
  (1, 'Alice Smith', 'Engineering', 75000.00, '2022-01-15'),
  (2, 'Bob Johnson', 'Marketing', 60000.00, '2021-03-20'),
  (3, 'Charlie Brown', 'Sales', 80000.00, '2023-06-01'),
  (4, 'Diana Prince', 'Engineering', 90000.00, '2020-11-10');



In [0]:
%sql
DESCRIBE DETAIL databricks_practice.inputdb.employees_vac

In [0]:
from delta.tables import DeltaTable

deltatable = DeltaTable.forName(spark,"databricks_practice.inputdb.employees_vac")

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

dry_run_df = deltatable.vacuum(0, dryRun=True) # 0 hours retention
dry_run_df.show(truncate=False)